# 🎯 SAM-Based Object Extractor
### Pixel-level object extraction from video using Meta's Segment Anything Model (SAM)

---

**Pipeline Overview:**
1. Upload a video at runtime
2. Extract all frames using OpenCV
3. Click once on the object in the first frame
4. SAM extracts the object from every frame with pixel-level precision
5. Save RGBA PNGs (transparent background) + binary masks

> ⚠️ **Before running:** Go to `Runtime → Change runtime type → GPU` (T4 or better recommended)

---

## Cell 1 — Install Dependencies
Run once per Colab session. Installs OpenCV, Matplotlib, Pillow, tqdm, and Meta's `segment-anything`.

In [ ]:
# %%capture  ← Uncomment to suppress verbose pip output

import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("opencv-python-headless", "matplotlib", "Pillow", "tqdm")
pip_install("git+https://github.com/facebookresearch/segment-anything.git")

print("✅ All packages installed.")

## Cell 2 — Imports

In [ ]:
import os
import sys
import shutil
import zipfile
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
from segment_anything import sam_model_registry, SamPredictor

# Google Colab file-upload helper
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("✅ Running inside Google Colab.")
except ImportError:
    IN_COLAB = False
    print("⚠️  Not in Colab — upload step will ask for a local file path.")

print("✅ All imports successful.")

## Cell 3 — Configuration
Edit these values to change model size, frame sampling rate, or prompt style.

In [ ]:
CFG = {
    # ── SAM model ────────────────────────────────────────────────────────────
    # "vit_h" = most accurate (~2.5 GB) | "vit_l" = balanced | "vit_b" = fastest
    "model_type"      : "vit_h",
    "checkpoint_url"  : "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
    "checkpoint_name" : "sam_vit_h_4b8939.pth",

    # ── Output directories ───────────────────────────────────────────────────
    "dir_frames"  : "frames",
    "dir_masks"   : "masks",
    "dir_output"  : "output",

    # ── Frame extraction ─────────────────────────────────────────────────────
    # 1 = every frame | 2 = every other frame | 5 = every 5th frame (faster)
    "frame_skip"  : 1,

    # ── Prompt mode ──────────────────────────────────────────────────────────
    # "point" → single left-click on object centre
    # "box"   → click-and-drag rectangle around object
    "prompt_mode" : "point",

    # ── Mask selection ───────────────────────────────────────────────────────
    # "largest"  → pick mask covering most pixels (good for dominant objects)
    # "best_iou" → use SAM's own confidence/IoU score
    "mask_strategy": "largest",
}

print("✅ Configuration set.")
for k, v in CFG.items():
    print(f"   {k:<20} : {v}")

## Cell 4 — GPU Check & Directory Setup

In [ ]:
def check_gpu() -> torch.device:
    """Verify GPU availability and return the appropriate torch device."""
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"🟢 GPU detected : {name}")
        print(f"   VRAM         : {vram:.1f} GB")
        return torch.device("cuda")
    else:
        print("🟡 No GPU found — running on CPU (will be slow for ViT-H).")
        print("   → Go to Runtime ▸ Change runtime type ▸ GPU and re-run.")
        return torch.device("cpu")


def make_dirs(*dirs: str) -> None:
    """Create clean output directories, removing any stale data."""
    for d in dirs:
        p = Path(d)
        if p.exists():
            shutil.rmtree(p)
        p.mkdir(parents=True, exist_ok=True)
    print(f"📁 Directories created: {', '.join(dirs)}")


# Run immediately
DEVICE = check_gpu()
make_dirs(CFG["dir_frames"], CFG["dir_masks"], CFG["dir_output"])

## Cell 5 — Upload Video & Extract Frames

In [ ]:
def upload_video() -> str:
    """
    Prompt the user to upload a video through Colab's file picker.
    Returns the path to the uploaded file.
    """
    if IN_COLAB:
        print("📤 Please upload your video file when the dialog appears …")
        uploaded = colab_files.upload()
        if not uploaded:
            raise RuntimeError("No file was uploaded. Please try again.")
        video_path = list(uploaded.keys())[0]
    else:
        video_path = input("Enter the full path to your video file: ").strip()
        if not Path(video_path).exists():
            raise FileNotFoundError(f"File not found: {video_path}")

    print(f"🎬 Video received : {video_path}")
    return video_path


def extract_frames(video_path: str, output_dir: str, frame_skip: int = 1) -> list:
    """
    Extract frames from a video file using OpenCV.

    Args:
        video_path  : Path to the source video.
        output_dir  : Directory where PNG frames will be saved.
        frame_skip  : Save every Nth frame (1 = all frames).

    Returns:
        Sorted list of saved frame file paths.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"\n📹 Video info:")
    print(f"   File         : {Path(video_path).name}")
    print(f"   Resolution   : {width}×{height}")
    print(f"   FPS          : {fps:.2f}")
    print(f"   Total frames : {total_frames}")
    print(f"   Frame skip   : every {frame_skip} frame(s)")

    saved_paths = []
    frame_idx   = 0
    saved_count = 0

    pbar = tqdm(total=total_frames, desc="Extracting frames", unit="fr")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_skip == 0:
            out_path = str(Path(output_dir) / f"frame_{saved_count:05d}.png")
            cv2.imwrite(out_path, frame)
            saved_paths.append(out_path)
            saved_count += 1
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    print(f"✅ {saved_count} frames extracted → /{output_dir}/")
    return sorted(saved_paths)


# Run upload and extraction
VIDEO_PATH  = upload_video()
FRAME_PATHS = extract_frames(VIDEO_PATH, CFG["dir_frames"], CFG["frame_skip"])

if not FRAME_PATHS:
    raise RuntimeError("No frames were extracted. Check that the video file is valid.")

## Cell 6 — Download SAM Checkpoint
This downloads the ViT-H checkpoint (~2.5 GB) once. Subsequent runs reuse the cached file.

In [ ]:
def download_checkpoint(url: str, name: str) -> str:
    """Download the SAM checkpoint if not already present locally."""
    if Path(name).exists():
        size_mb = Path(name).stat().st_size / 1e6
        print(f"✅ Checkpoint found locally ({size_mb:.0f} MB) — skipping download.")
        return name

    print(f"⬇️  Downloading SAM checkpoint ({name}) …")
    print("   This is ~2.5 GB — grab a coffee ☕")

    def _reporthook(count, block_size, total_size):
        pct = count * block_size * 100 / max(total_size, 1)
        print(f"\r   Progress: {min(pct, 100):.1f}%", end="", flush=True)

    urllib.request.urlretrieve(url, name, reporthook=_reporthook)
    print(f"\n✅ Download complete — {Path(name).stat().st_size / 1e6:.0f} MB saved.")
    return name


CKPT_PATH = download_checkpoint(CFG["checkpoint_url"], CFG["checkpoint_name"])

## Cell 7 — Load SAM Model

In [ ]:
def load_sam(model_type: str, checkpoint: str, device: torch.device) -> SamPredictor:
    """
    Instantiate the SAM model and return a SamPredictor ready for inference.

    Args:
        model_type : "vit_h", "vit_l", or "vit_b"
        checkpoint : Path to the .pth weights file
        device     : torch.device (cuda / cpu)

    Returns:
        SamPredictor instance
    """
    print(f"\n🤖 Loading SAM ({model_type}) from {checkpoint} …")
    sam = sam_model_registry[model_type](checkpoint=checkpoint)
    sam.to(device=device)
    sam.eval()
    predictor = SamPredictor(sam)
    print("✅ SAM model loaded and ready.")
    return predictor


PREDICTOR = load_sam(CFG["model_type"], CKPT_PATH, DEVICE)

## Cell 8 — Interactive Prompt Collection

The first frame will be displayed below.

- **Point mode** → Left-click once on the **centre of the object** you want to extract.
- **Box mode** → Click and drag a rectangle around the object.

The figure closes automatically after your input.

In [ ]:
# Module-level state for matplotlib interactive callbacks
_prompt_data: dict = {}


def _onclick_point(event):
    """Callback: record a single left-click coordinate and close the figure."""
    if event.inaxes and event.button == 1:
        _prompt_data["point"] = (int(event.xdata), int(event.ydata))
        plt.close()


def _onpress_box(event):
    """Callback: record drag start position."""
    if event.inaxes and event.button == 1:
        _prompt_data["box_start"] = (int(event.xdata), int(event.ydata))


def _onrelease_box(event):
    """Callback: record drag end, compute box, and close the figure."""
    if event.inaxes and event.button == 1 and "box_start" in _prompt_data:
        x0, y0 = _prompt_data["box_start"]
        x1, y1 = int(event.xdata), int(event.ydata)
        _prompt_data["box"] = (min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))
        plt.close()


def collect_prompt(first_frame_path: str, mode: str = "point") -> dict:
    """
    Display the first frame and collect a user prompt interactively.

    Args:
        first_frame_path : Path to the first extracted frame PNG.
        mode             : "point" or "box"

    Returns:
        Dict with key "point" (x, y) OR "box" (x0, y0, x1, y1).
    """
    _prompt_data.clear()

    img_bgr = cv2.imread(first_frame_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(img_rgb)
    ax.axis("off")

    if mode == "point":
        ax.set_title(
            "🖱️  LEFT-CLICK on the centre of the object you want to extract.\n"
            "The window will close automatically after your click.",
            fontsize=11, color="navy"
        )
        fig.canvas.mpl_connect("button_press_event", _onclick_point)
    else:
        ax.set_title(
            "🖱️  Click-and-drag a rectangle AROUND the target object.\n"
            "Release the mouse to confirm. The window closes automatically.",
            fontsize=11, color="navy"
        )
        fig.canvas.mpl_connect("button_press_event",   _onpress_box)
        fig.canvas.mpl_connect("button_release_event", _onrelease_box)

    plt.tight_layout()
    plt.show()

    # Validate
    if mode == "point" and "point" not in _prompt_data:
        raise RuntimeError("No point selected. Re-run this cell and click on the object.")
    if mode == "box" and "box" not in _prompt_data:
        raise RuntimeError("No box drawn. Re-run this cell and drag a rectangle.")

    if mode == "point":
        x, y = _prompt_data["point"]
        print(f"✅ Prompt recorded — Point: ({x}, {y})")
    else:
        print(f"✅ Prompt recorded — Box: {_prompt_data['box']}")

    return dict(_prompt_data)


# Collect prompt from the user
print(f"📌 Prompt mode: '{CFG['prompt_mode']}'")
PROMPT = collect_prompt(FRAME_PATHS[0], mode=CFG["prompt_mode"])

## Cell 9 — SAM Inference & Mask Application Helpers

In [ ]:
def predict_mask(
    predictor  : SamPredictor,
    image_rgb  : np.ndarray,
    prompt     : dict,
    strategy   : str = "largest"
) -> np.ndarray:
    """
    Run SAM on one frame and select the best binary mask.

    Args:
        predictor  : Initialized SamPredictor.
        image_rgb  : RGB uint8 numpy array (H, W, 3).
        prompt     : Dict from collect_prompt() — contains 'point' or 'box'.
        strategy   : 'largest' or 'best_iou'.

    Returns:
        Binary mask as bool numpy array (H, W).
    """
    predictor.set_image(image_rgb)

    point_coords = point_labels = input_box = None

    if "point" in prompt:
        x, y         = prompt["point"]
        point_coords = np.array([[x, y]])
        point_labels = np.array([1])       # 1 = foreground

    if "box" in prompt:
        x0, y0, x1, y1 = prompt["box"]
        input_box       = np.array([x0, y0, x1, y1])

    masks, iou_scores, _ = predictor.predict(
        point_coords     = point_coords,
        point_labels     = point_labels,
        box              = input_box,
        multimask_output = True,     # request up to 3 candidate masks
    )

    # Select best mask according to strategy
    if strategy == "best_iou":
        best_idx = int(np.argmax(iou_scores))
    else:  # "largest"
        best_idx = int(np.argmax([m.sum() for m in masks]))

    return masks[best_idx]


def apply_mask_rgba(image_bgr: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """
    Composite the binary mask onto the frame → RGBA image.
    Background pixels get alpha = 0 (fully transparent).

    Args:
        image_bgr : BGR frame from OpenCV (H, W, 3).
        mask      : Boolean mask (H, W).

    Returns:
        RGBA numpy array (H, W, 4) as uint8.
    """
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    alpha     = mask.astype(np.uint8) * 255        # 255 inside, 0 outside
    rgba      = np.dstack([image_rgb, alpha])
    return rgba.astype(np.uint8)


print("✅ Inference helpers defined.")

## Cell 10 — Process All Frames
This is the main inference loop. SAM re-encodes each frame and applies the saved prompt to produce a pixel-perfect mask.

In [ ]:
def process_all_frames(
    frame_paths : list,
    predictor   : SamPredictor,
    prompt      : dict,
    masks_dir   : str,
    output_dir  : str,
    strategy    : str = "largest"
) -> None:
    """
    Iterate over every extracted frame, run SAM, and save:
      - Binary mask PNG  → masks_dir/
      - RGBA object PNG  → output_dir/

    Args:
        frame_paths : Sorted list of frame file paths.
        predictor   : Loaded SamPredictor.
        prompt      : Prompt dict (point or box).
        masks_dir   : Destination for binary mask images.
        output_dir  : Destination for RGBA extracted-object images.
        strategy    : Mask selection strategy.
    """
    total = len(frame_paths)
    print(f"\n🔄 Processing {total} frames with SAM …")

    for idx, frame_path in enumerate(tqdm(frame_paths, desc="SAM inference", unit="fr")):
        stem = Path(frame_path).stem

        # Load frame
        img_bgr = cv2.imread(frame_path)
        if img_bgr is None:
            print(f"⚠️  Could not read {frame_path} — skipping.")
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Predict mask
        try:
            mask = predict_mask(predictor, img_rgb, prompt, strategy)
        except Exception as exc:
            print(f"⚠️  SAM failed on {stem}: {exc} — skipping.")
            continue

        # Save binary mask (white = object, black = background)
        mask_img = mask.astype(np.uint8) * 255
        cv2.imwrite(str(Path(masks_dir) / f"{stem}_mask.png"), mask_img)

        # Save RGBA output with transparent background
        rgba    = apply_mask_rgba(img_bgr, mask)
        pil_img = Image.fromarray(rgba, mode="RGBA")
        pil_img.save(str(Path(output_dir) / f"{stem}_extracted.png"))

    print(f"\n✅ All frames processed.")
    print(f"   Masks   → /{masks_dir}/")
    print(f"   Objects → /{output_dir}/")


# Run the processing loop
process_all_frames(
    frame_paths = FRAME_PATHS,
    predictor   = PREDICTOR,
    prompt      = PROMPT,
    masks_dir   = CFG["dir_masks"],
    output_dir  = CFG["dir_output"],
    strategy    = CFG["mask_strategy"]
)

## Cell 11 — Preview First Result
Displays a 3-panel comparison: **Original | Binary Mask | Extracted Object**

In [ ]:
def preview_result(first_frame_path: str, masks_dir: str, output_dir: str) -> None:
    """
    Display a 3-panel figure:
      [Original frame | Binary mask | Extracted object on white background]
    """
    stem     = Path(first_frame_path).stem
    orig_bgr = cv2.imread(first_frame_path)
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)

    mask_path   = str(Path(masks_dir)  / f"{stem}_mask.png")
    output_path = str(Path(output_dir) / f"{stem}_extracted.png")

    mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    rgba_img = np.array(Image.open(output_path).convert("RGBA"))

    # Composite extracted object onto a white background for display
    white_bg     = np.ones_like(orig_rgb) * 255
    alpha_float  = rgba_img[:, :, 3:4].astype(float) / 255.0
    obj_on_white = (
        rgba_img[:, :, :3] * alpha_float + white_bg * (1 - alpha_float)
    ).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    panels = [
        (orig_rgb,     "Original Frame",               None),
        (mask_img,     "Binary Mask",                   "gray"),
        (obj_on_white, "Extracted Object (white BG)",   None),
    ]

    for ax, (img, title, cmap) in zip(axes, panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.axis("off")

    plt.suptitle("SAM Object Extraction — First Frame Preview", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig("preview_result.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("🖼️  Preview saved → preview_result.png")


preview_result(FRAME_PATHS[0], CFG["dir_masks"], CFG["dir_output"])

## Cell 12 — Run Summary

In [ ]:
def print_summary(video_path, frame_paths, device, cfg):
    """Print a clean run summary to the console."""
    n_masks   = len(list(Path(cfg["dir_masks"]).glob("*.png")))
    n_outputs = len(list(Path(cfg["dir_output"]).glob("*.png")))

    print("\n" + "═" * 60)
    print("  SAM OBJECT EXTRACTION — RUN SUMMARY")
    print("═" * 60)
    print(f"  Video file      : {Path(video_path).name}")
    gpu_name = torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'
    print(f"  GPU / Device    : {device} ({gpu_name})")
    print(f"  SAM model       : {cfg['model_type'].upper()}")
    print(f"  Prompt mode     : {cfg['prompt_mode']}")
    print(f"  Mask strategy   : {cfg['mask_strategy']}")
    print(f"  Frames extracted: {len(frame_paths)}")
    print(f"  Masks saved     : {n_masks}")
    print(f"  RGBA outputs    : {n_outputs}")
    print(f"\n  Output dirs:")
    print(f"    /frames/   — raw video frames (PNG)")
    print(f"    /masks/    — binary object masks (PNG)")
    print(f"    /output/   — transparent-background PNGs (RGBA)")
    print("═" * 60)
    print("  ✅ Pipeline completed successfully!")
    print("═" * 60)


print_summary(VIDEO_PATH, FRAME_PATHS, DEVICE, CFG)

## Cell 13 — Download Results
Zips all output directories and triggers a browser download via the Colab files API.

In [ ]:
zip_name = "sam_extraction_results.zip"

print(f"📦 Zipping outputs → {zip_name} …")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in [CFG["dir_frames"], CFG["dir_masks"], CFG["dir_output"]]:
        for f in sorted(Path(d).glob("*.png")):
            zf.write(f, arcname=str(f))
    # Also include the preview
    if Path("preview_result.png").exists():
        zf.write("preview_result.png")

zip_size = Path(zip_name).stat().st_size / 1e6
print(f"✅ Zip created: {zip_name} ({zip_size:.1f} MB)")

if IN_COLAB:
    print("⬇️  Starting download …")
    colab_files.download(zip_name)
else:
    print(f"📁 File saved locally: {Path(zip_name).resolve()}")

---

## 📋 Output Structure

```
/frames/          ← frame_00000.png … frame_NNNNN.png    (raw BGR frames)
/masks/           ← frame_00000_mask.png …               (binary: white = object)
/output/          ← frame_00000_extracted.png …          (RGBA, transparent BG)
preview_result.png ← side-by-side comparison of first frame
sam_extraction_results.zip ← all of the above, bundled
```

## ⚙️ Configuration Quick Reference

| Key | Default | Options |
|---|---|---|
| `model_type` | `vit_h` | `vit_h`, `vit_l`, `vit_b` |
| `frame_skip` | `1` | Any integer ≥ 1 |
| `prompt_mode` | `point` | `point`, `box` |
| `mask_strategy` | `largest` | `largest`, `best_iou` |

---
*Pipeline built with Meta SAM + OpenCV + Python. No training required.*